In [9]:
import os
import copy
import torch
import numpy as np
import pandas as pd

# Core utilities
from src.safety_evaluation import get_gamma
from src.safety_evaluation.utils.utils import (
    get_merged_calibration_result_path,
    get_merged_upb_calibration_result_path
)
from src.utils.utils import set_seeds

# Import our new shared module
from common import (
    plot_coverage_bounds_vs_target_coverage,
    plot_variability,
    plot_coverage_diff,
    plot_metrics,
    metrics_boxplots,
)


# Mapping dictionaries
LPB_DISPLAY_NAME_MAP = {
    'calibration_optimized_allocation': 'Static (Baseline)',
    'calibration_projected_optimization_platt_prob_allocation': 'Dynamic (Ours)',
}

UPB_DISPLAY_NAME_MAP = {
    'uncalibrated': 'Uncalibrated',
    'calibration_optimized_allocation': 'Optimized',
    'calibration_adaptive_optimized_allocation': 'Locally Adaptive',
    'calibration_projected_optimization_platt_prob_allocation': 'Ours',
}

### Define Pipeline Execution

In [7]:
def execute_evaluation(experiments_name, figures_dir, path_resolver_func, display_name_map_dict, config, bound_type="LPB"):
    base_results_dir = path_resolver_func(experiments_name)
    if not os.path.exists(base_results_dir):
        print(f"No results in path {os.path.abspath(base_results_dir)}")
        return
        
    all_df = pd.read_csv(os.path.join(base_results_dir, "all_df.csv"))
    if len(all_df) == 0:
        print("DataFrame exists but is empty.")
        return

    exp_fig_dir = os.path.abspath(os.path.join(figures_dir, experiments_name))
    if os.name == 'nt' and not exp_fig_dir.startswith('\\\\?\\'):
        exp_fig_dir = '\\\\?\\' + exp_fig_dir
    os.makedirs(exp_fig_dir, exist_ok=True)

    # Filter mapping to match available data
    active_display_map = {k: v for k, v in display_name_map_dict.items() if k in all_df['calibration_name'].unique()}
    active_color_map = {k: v for k, v in COLOR_MAP.items() if k in active_display_map.values()}
    legend_order = list(active_color_map.keys())

    # Map dataframe entries
    all_df = all_df[(~all_df['calibration_name'].isna()) & (~all_df['seed'].isna())]
    all_df_mapped = all_df.copy()
    all_df_mapped['calibration_name'] = all_df_mapped['calibration_name'].map(active_display_map)

    # 1. Base Coverage Plots
    plot_coverage_bounds_vs_target_coverage(all_df_mapped, active_color_map, legend_order, bound_type=bound_type, save_dir=exp_fig_dir)
    plot_variability(all_df_mapped, active_color_map, legend_order, exp_fig_dir)
    plot_coverage_diff(all_df_mapped, active_color_map, legend_order, exp_fig_dir)
    

    # 3. Metric Overviews
    df_baseline = all_df[all_df['calibration_name'].isin(active_display_map.keys())].copy()
    metrics_boxplots(df_baseline, active_display_map, active_color_map, exp_fig_dir)
    
    total_budget = config['budget_per_sample'] * config['cal_size']
    plot_metrics(all_df_mapped, active_color_map, legend_order, total_budget, exp_fig_dir)

    # 4. Text Output Comparison
    df = copy.deepcopy(all_df_mapped)
    available_methods = df['calibration_name'].unique()
    if 'Adaptive' in available_methods and 'Optimized' in available_methods:
        df['target_coverage'] = np.round(df['target_coverage'] * 100, 2)
        df['coverage'] = np.round(df['coverage'] * 100, 2)
        df['coverage_diff'] = (df['coverage'] - df['target_coverage']).abs()
        target_cov_df = df[(df['target_coverage'] == 90)]

        for method, label in [('Optimized', 'Optimized'), ('Adaptive', 'Adaptive'), 
                              ('Proj. Opt.', 'Projection Optimization'), 
                              ('New (0.1) (with diff)', 'New (0.1)'), 
                              ('New (0.95) (with diff)', 'New (0.95)')]:
            if method in target_cov_df['calibration_name'].values:
                sub_df = target_cov_df[target_cov_df['calibration_name'] == method]
                print(f"{label}: {sub_df['coverage_diff'].mean():.3f} ({sub_df['coverage_diff'].std():.2f})")

## LPB Runner

Load the LPB (Lower Probability Bound) coverage data for the predefined dataset groupings, map them to standard display names, and plot the resulting coverage boxplots to evaluate the conservatism of our bound.

In [8]:
# Configure Experiment Parameters
config = {
    'dataset_name': '',
    'dataset_setup': '',
    'budget_per_sample': 1.0,
    'cal_size': 4000,
    'tau_prior': 0.56,
    'is_real': True,
    'device': 'cuda:0' if torch.cuda.is_available() else 'cpu'
}

set_seeds(0)

# Calculate parameters
m_upper_bound = 200 if config['is_real'] else 20
gamma = get_gamma(m_upper_bound, config['budget_per_sample'])
print(f"Budget per sample: {config['budget_per_sample']}, Gamma: {gamma}, Upper Bound: {m_upper_bound}")

experiment_signature = f"{config['dataset_name']}_{config['dataset_setup']}_{config['budget_per_sample']}_{config['cal_size']}_{config['tau_prior']}_{gamma}"

# Run Pipeline
print("\n--- Running LPB Evaluation ---")
execute_evaluation(
    experiments_name=experiment_signature,
    figures_dir="./figures/calibration_results/",
    path_resolver_func=get_merged_calibration_result_path,
    display_name_map_dict=LPB_DISPLAY_NAME_MAP,
    config=config,
    bound_type="LPB"
)



Budget per sample: 1.0, Gamma: 200.0, Upper Bound: 200

--- Running LPB Evaluation ---
No results in path /Users/shai.feldman/Documents/Projects/dapro_refactor/notebooks/results/merged_calibration_dfs/__1.0_4000_0.56_200.0

--- Running UPB Evaluation ---
No results in path /Users/shai.feldman/Documents/Projects/dapro_refactor/notebooks/results/merged_upb_calibration_dfs/__1.0_4000_0.56_200.0

Evaluation pipeline finished successfully.


## UPB Runner

Load and process the UPB (Upper Probability Bound) coverage results for the AutoIF configurations, and generate the corresponding boxplots.

In [ ]:
# Configure Experiment Parameters
config = {
    'dataset_name': '',
    'dataset_setup': '',
    'budget_per_sample': 1.0,
    'cal_size': 4000,
    'tau_prior': 0.56,
    'is_real': True,
    'device': 'cuda:0' if torch.cuda.is_available() else 'cpu'
}

set_seeds(0)

# Calculate parameters
m_upper_bound = 200 if config['is_real'] else 20
gamma = get_gamma(m_upper_bound, config['budget_per_sample'])
print(f"Budget per sample: {config['budget_per_sample']}, Gamma: {gamma}, Upper Bound: {m_upper_bound}")

experiment_signature = f"{config['dataset_name']}_{config['dataset_setup']}_{config['budget_per_sample']}_{config['cal_size']}_{config['tau_prior']}_{gamma}"


print("\n--- Running UPB Evaluation ---")
execute_evaluation(
    experiments_name=experiment_signature,
    figures_dir="./figures/calibration_upb_results/",
    path_resolver_func=get_merged_upb_calibration_result_path,
    display_name_map_dict=UPB_DISPLAY_NAME_MAP,
    config=config,
    bound_type="UPB"
)

print("\nEvaluation pipeline finished successfully.")

## Safety metrics configurations

Define the target budgets and dictionary mappings that link raw dataset run filenames with clean display names (e.g., specific LLMs like Llama3.1, Phi4Mini, Qwen2.5) across safety categories like Toxicity, Red-Teaming, Hallucination, and AutoIF.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Global safety plot styling overrides
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.size': 28, 'axes.titlesize': 28, 'axes.labelsize': 28,
    'xtick.labelsize': 28, 'ytick.labelsize': 28, 'legend.fontsize': 28, 'legend.title_fontsize': 28
})

DISPLAY_NAME_MAP_BASELINES = {
    'UnweightedUniformBudgetAllocator': 'Unweighted Uniform',
    'UniformBudgetAllocator': 'Uniform',
    'optimized': 'Optimized',
    'projected_optimization_platt_prob': 'Ours',
}

DISPLAY_NAME_MAP_GROUPED = {
    'optimized': 'Optimized',
    'new_diff=True_first_step_budget=0.1': 'Greedy (0.1)',
    'adaptive_optimized': 'Locally Adaptive',
    f'projected_optimization_platt_prob': 'DAPRO',
}

# Update existing COLOR_MAP from Cell 1 with the new combinations
COLOR_MAP.update({
    'Naive': 'tab:red',
    'Uniform': 'tab:orange',
    'Unweighted Uniform': 'tab:gray',
    'Proj. Opt. (Prob)': 'tab:green',
    'DAPRO': 'tab:green',
})

METRICS_INFO = {
    'estimated_cjr': {'title': 'Estimated SER', 'ylabel': 'SER (%)', 'is_paper': True},
    'abs_diff_cjr': {'title': 'Abs. Error: SER', 'ylabel': r'$|\widehat{SER} - SER^*|$ (%)', 'is_paper': True},
    'estimated_rmttu': {'title': 'Estimated RMTTS', 'ylabel': 'RMTTS (Iterations)', 'is_paper': True},
    'abs_diff_rmttu': {'title': 'Abs. Error: RMTTS', 'ylabel': r'$|\widehat{RMTTS} - RMTTS^*|$', 'is_paper': True},
    'total_compute_iterations': {'title': 'Total Compute', 'ylabel': 'Iterations', 'is_paper': False},
    'budget_per_sample': {'title': 'Budget Per Sample', 'ylabel': 'Budget', 'is_paper': False},
    'observed_jailbreaks': {'title': 'Observed Events', 'ylabel': 'Event Count', 'is_paper': False},
}

VARIANCE_METRICS_INFO = {
    'estimated_cjr': {'title': 'Variance of Est. SER', 'ylabel': r'$\text{Var}[\widehat{SER}]\text{ }[(\%)^2]$ ', 'is_paper': True},
    'estimated_rmttu': {'title': 'Variance of Est. RMTTS', 'ylabel': r'$\text{Var}[\widehat{RMTTS}]\text{ }[\text{Iter.}^2]$', 'is_paper': True}
}

# --- GROUPED DATASET MAPS ---
toxicity_target_budget = 20
toxicity_map = {
    f'dataset_toxicity_attack_toxic_attack_qwen25_14b_instruct_lm_target_llama_31_8B_instruct_judge_detoxify_{float(toxicity_target_budget)}_safety_metrics': "Llama3.1",
    f'dataset_toxicity_attack_toxic_attack_qwen25_14b_instruct_lm_target_mini_phi_4_instruct_judge_detoxify_{float(toxicity_target_budget)}_safety_metrics': "Phi4Mini",
    f'dataset_toxicity_attack_toxic_attack_qwen25_14b_instruct_lm_target_qwen25_14b_instruct_judge_detoxify_{float(toxicity_target_budget)}_safety_metrics': "Qwen2.5",
    f'dataset_toxicity_attack_toxic_attack_qwen25_14b_instruct_lm_target_gemma3_4b_it_judge_detoxify_{float(toxicity_target_budget)}_safety_metrics': "Gemma3",
}

redteam_target_budget = 20
redteam_map = {
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_llama_31_8B_instruct_judge_llm-judge_qwen25_14b_instruct_{float(redteam_target_budget)}_safety_metrics': "Llama3.1",
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_mini_phi_4_instruct_judge_llm-judge_qwen25_14b_instruct_{float(redteam_target_budget)}_safety_metrics': "Phi4Mini",
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_qwen25_14b_instruct_judge_llm-judge_qwen25_14b_instruct_{float(redteam_target_budget)}_safety_metrics': "Qwen2.5",
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_gemma3_4b_it_judge_llm-judge_qwen25_14b_instruct_{float(redteam_target_budget)}_safety_metrics': "Gemma3",
}

redteam_llam_guard_target_budget = 20
redteam_llam_guard_map = {
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_llama_31_8B_instruct_judge_llama_guard_{float(redteam_llam_guard_target_budget)}_safety_metrics': "Llama3.1",
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_mini_phi_4_instruct_judge_llama_guard_{float(redteam_llam_guard_target_budget)}_safety_metrics': "Phi4Mini",
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_qwen25_14b_instruct_judge_llama_guard_{float(redteam_llam_guard_target_budget)}_safety_metrics': "Qwen2.5",
    f'dataset_red_team_attack_default_attack_qwen25_14b_instruct_lm_target_gemma3_4b_it_judge_llama_guard_{float(redteam_llam_guard_target_budget)}_safety_metrics': "Gemma3",
}

hallucination_target_budget = 20
hallucination_map = {
    f'dataset_hallucination3_attack_hallucination_attack_qwen25_14b_instruct_lm_target_llama_31_8B_instruct_judge_llm-judge_qwen25_14b_instruct_{float(hallucination_target_budget)}_safety_metrics': "Llama3.1",
    f'dataset_hallucination3_attack_hallucination_attack_qwen25_14b_instruct_lm_target_mini_phi_4_instruct_judge_llm-judge_qwen25_14b_instruct_{float(hallucination_target_budget)}_safety_metrics': "Phi4Mini",
    f'dataset_hallucination3_attack_hallucination_attack_qwen25_14b_instruct_lm_target_qwen25_14b_instruct_judge_llm-judge_qwen25_14b_instruct_{float(hallucination_target_budget)}_safety_metrics': "Qwen2.5",
}

autoif_target_budget = 20
autoif_map = {
    f'dataset_autoif_attack_autoif_helper_qwen25_14b_instruct_lm_target_llama_31_8B_instruct_judge_autoif_{float(hallucination_target_budget)}_safety_metrics': "Llama3.1",
    f'dataset_autoif_attack_autoif_helper_qwen25_14b_instruct_lm_target_qwen25_14b_instruct_judge_autoif_{float(hallucination_target_budget)}_safety_metrics': "Qwen2.5",
}

## Safety metrics visualizations

Iterate through the grouped datasets defined above, apply IPCW weighting via `MetricsEngine`, and generate visual summaries (boxplots/bar charts) for safety metrics like Cumulative Jailbreak Rate and Cost per Jailbreak.

In [ ]:
# 1. Configuration
figures_dir = 'figures'
safety_config = {
    'dataset_name': '',
    'dataset_setup': '',
    'budget_per_sample': 40.0,
    'cal_size': 4000,
    'tau_prior': 0.56,
}

experiments_name = f"{safety_config['dataset_name']}_{safety_config['dataset_setup']}_{safety_config['budget_per_sample']}_safety_metrics"
csv_path = os.path.join(get_merged_metric_calibration_result_path(experiments_name), "all_df.csv")

# 2. Run Single Plot
if os.path.exists(csv_path):
    print(f"\n--- Running Single Plot for {experiments_name} ---")
    df = pd.read_csv(csv_path)
    for col in METRICS_INFO.keys():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    plot_safety_metrics_single(
        df=df, experiments_name=safety_config['dataset_name'], 
        budget_per_sample=safety_config['budget_per_sample'], 
        display_name_map=DISPLAY_NAME_MAP_BASELINES, 
        color_map=COLOR_MAP, metrics_info=METRICS_INFO, 
        variance_metrics_info=VARIANCE_METRICS_INFO, figures_dir=figures_dir
    )
else:
    print(f"Skipping single run: Could not find merged dataset at {os.path.abspath(csv_path)}")

# 3. Run Grouped Plots
print("\n--- Running Grouped Dataset Plots ---")

# Optional: Uncomment the blocks below if you wish to run all datasets
# df_tox = load_and_prep_grouped_data(toxicity_map, DISPLAY_NAME_MAP_GROUPED, "toxicity", METRICS_INFO, get_merged_metric_calibration_result_path)
# generate_grouped_plots(df_tox, figures_dir, "toxicity", COLOR_MAP, toxicity_target_budget, DISPLAY_NAME_MAP_GROUPED, METRICS_INFO, VARIANCE_METRICS_INFO)

# df_red_qwen = load_and_prep_grouped_data(redteam_map, DISPLAY_NAME_MAP_GROUPED, "red_team_qwen", METRICS_INFO, get_merged_metric_calibration_result_path)
# generate_grouped_plots(df_red_qwen, figures_dir, "red_team_qwen", COLOR_MAP, redteam_target_budget, DISPLAY_NAME_MAP_GROUPED, METRICS_INFO, VARIANCE_METRICS_INFO)

# df_red_llama = load_and_prep_grouped_data(redteam_llam_guard_map, DISPLAY_NAME_MAP_GROUPED, "red_team_llama_guard", METRICS_INFO, get_merged_metric_calibration_result_path)
# generate_grouped_plots(df_red_llama, figures_dir, "red_team_llama_guard", COLOR_MAP, redteam_llam_guard_target_budget, DISPLAY_NAME_MAP_GROUPED, METRICS_INFO, VARIANCE_METRICS_INFO)

# df_hallucination = load_and_prep_grouped_data(hallucination_map, DISPLAY_NAME_MAP_GROUPED, "hallucination", METRICS_INFO, get_merged_metric_calibration_result_path)
# generate_grouped_plots(df_hallucination, figures_dir, "hallucination", COLOR_MAP, hallucination_target_budget, DISPLAY_NAME_MAP_GROUPED, METRICS_INFO, VARIANCE_METRICS_INFO)

df_autoif = load_and_prep_grouped_data(
    autoif_map, DISPLAY_NAME_MAP_GROUPED, "autoif", 
    METRICS_INFO, get_merged_metric_calibration_result_path
)
generate_grouped_plots(
    df_autoif, figures_dir, "autoif", COLOR_MAP, autoif_target_budget, 
    DISPLAY_NAME_MAP_GROUPED, METRICS_INFO, VARIANCE_METRICS_INFO
)